# 06 — PP-OCRv4 Scene2D Typography Fine-Tune, D-FINE-S Mixed-Precision FP16/FP32 & Pure-COCO RTMPose-S

1. **D-FINE-S Mixed Precision:** Casts only the HGNetv2 backbone of `dfine_s_coco.onnx` to FP16 while keeping the sensitive DETR box head in FP32 (fixing the FP16 box underflow caught in Phase 3; cuts 41.5 MB -> ~20 MB and +123 MB RSS -> ~60 MB).
2. **PP-OCRv4 Rec Fine-Tune:** Fine-tunes the 10 MB recognizer on 20k Infinite `Scene2DNode` stylized text renders (outlines, gradients, warps) to drop CER from 6.2% -> < 2% at identical latency.
3. **Pure-COCO RTMPose-S:** Trains Apache-2.0 CSPNeXt+SimCC from scratch on MS COCO Keypoints only (zero AI Challenger / MPII taint).


In [ ]:
!pip -q install -U torch torchvision timm onnx onnxslim onnxruntime lpips pillow numpy tqdm huggingface_hub
import os, hashlib, torch, onnx, onnxslim, onnxruntime as ort
print("PyTorch:", torch.__version__, "CUDA:", torch.cuda.is_available())


In [ ]:
# 1. Mixed-precision D-FINE-S (FP16 backbone + FP32 DETR decoder/box head)
import urllib.request, shutil
from onnxruntime.transformers.float16 import convert_float_to_float16
from google.colab import drive
drive.mount('/content/drive')

url = 'https://huggingface.co/onnx-community/dfine_s_coco-ONNX/resolve/main/onnx/model.onnx'
urllib.request.urlretrieve(url, '/content/dfine_s_coco_fp32.onnx')
m = onnx.load('/content/dfine_s_coco_fp32.onnx')
fp32_nodes = [n.name for n in m.graph.node if any(k in n.name.lower() for k in ('decoder', 'bbox', 'box', 'class', 'score', 'softmax', 'sigmoid', 'layernorm', 'norm'))]
m_mix = convert_float_to_float16(m, keep_io_types=True, node_block_list=fp32_nodes, disable_shape_infer=True)

# Topologically sort graph nodes (convert_float_to_float16 can place input Cast nodes after consumers)
def topo_sort(model):
    inits = {i.name for i in model.graph.initializer} | {i.name for i in model.graph.input}
    nodes = list(model.graph.node)
    produced = set(inits)
    ordered = []
    remaining = nodes
    while remaining:
        nxt, stuck = [], []
        for n in remaining:
            if all((inp == '' or inp in produced) for inp in n.input):
                nxt.append(n)
            else:
                stuck.append(n)
        if not nxt:
            ordered.extend(stuck)
            break
        for n in nxt:
            ordered.append(n)
            produced.update(o for o in n.output if o)
        remaining = stuck
    del model.graph.node[:]
    model.graph.node.extend(ordered)
    return model

m_mix = topo_sort(m_mix)
m_mix = onnxslim.slim(m_mix)
onnx.checker.check_model(m_mix)
onnx.save(m_mix, '/content/dfine_s_coco_mixed.onnx')
# Quick runtime sanity check on random input
sess = ort.InferenceSession('/content/dfine_s_coco_mixed.onnx', providers=['CPUExecutionProvider'])
import numpy as np
lo, bx = sess.run(None, {'pixel_values': np.random.rand(1, 3, 640, 640).astype(np.float32)})
print("Sanity boxes min/max:", float(bx.min()), float(bx.max()), "nan:", bool(np.isnan(bx).any()))
data = open('/content/dfine_s_coco_mixed.onnx', 'rb').read()
print(f"Mixed D-FINE-S: {len(data)/1e6:.2f} MB  sha256={hashlib.sha256(data).hexdigest()}")
shutil.copy2('/content/dfine_s_coco_mixed.onnx', '/content/drive/MyDrive/infinite_ml/dfine_s_coco_mixed.onnx')
print("Saved to Drive: /content/drive/MyDrive/infinite_ml/dfine_s_coco_mixed.onnx")
